In [1]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.naive_bayes import MultinomialNB
import string
from nltk.corpus import stopwords
from collections import Counter
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score
import scipy.sparse as sp

# Load the data

In [2]:
## Read Data
df = pd.read_csv("dataset/training_data.csv", sep="\t", header=None, names=["label", "text"])

In [3]:
# Coppying data:
data = df.copy()

# Inspect the data

In [4]:
# Inspect the data
data.head()

,label,text
0,0,donald trump sends out embarrassing new year‚s...
1,0,drunk bragging trump staffer started russian c...
2,0,sheriff david clarke becomes an internet joke ...
3,0,trump is so obsessed he even has obama‚s name ...
4,0,pope francis just called out donald trump duri...


In [5]:
data.tail()

,label,text
34147,1,tears in rain as thais gather for late king's ...
34148,1,pyongyang university needs non-u.s. teachers a...
34149,1,philippine president duterte to visit japan ah...
34150,1,japan's abe may have won election\tbut many do...
34151,1,demoralized and divided: inside catalonia's po...


It seems the rows are grouped by label.

In [6]:
data.shape

(34152, 2)

In [7]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34152 entries, 0 to 34151
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   34152 non-null  int64 
 1   text    34152 non-null  object
dtypes: int64(1), object(1)
memory usage: 533.8+ KB


The dataset is composed by 34152 rows and 2 columns. The label column contains only integers (0 and 1) and the text column is of object type.

In [8]:
data.isna().sum()

label    0
text     0
dtype: int64

In [9]:
data.duplicated().sum()

np.int64(1946)

In [10]:
data['text'].duplicated().sum()

np.int64(1946)

There are no rows with missing information and there are 1946 duplicated entries, which should be removed.

# Pre-process the data

In [11]:
# Remove duplicates
data_clean = data.drop_duplicates()

print(f"Original shape: {data.shape}")
print(f"Clean data shape: {data_clean.shape}")

Original shape: (34152, 2)
Clean data shape: (32206, 2)


In [12]:
data_clean.duplicated().sum()

np.int64(0)

## Clean the text

In [13]:
# Create a function to clean the text 
def clean_text(text):
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text) # Remove special characters
    text = re.sub(r"\s+[a-zA-Z]\s+", " ", text) # Remove all single characters
    text = re.sub(r"\^[a-zA-Z]\s+", " ", text) # Remove single characters from the start
    text = re.sub(r"\s+", " ", text) # Substitute multiple spaces with a single space
    text = text.lower() # Convert to lowercase

    return text

Numbers were not removed considering they might give context to the model.

In [14]:
data_clean['text_clean'] = data_clean['text'].apply(clean_text)
data_clean.head()

/var/folders/ns/tz5d_bp51j59j7fmbrqg1d1h0000gn/T/ipykernel_20681/65238994.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_clean['text_clean'] = data_clean['text'].apply(clean_text)


,label,text,text_clean
0,0,donald trump sends out embarrassing new year‚s...,donald trump sends out embarrassing new year e...
1,0,drunk bragging trump staffer started russian c...,drunk bragging trump staffer started russian c...
2,0,sheriff david clarke becomes an internet joke ...,sheriff david clarke becomes an internet joke ...
3,0,trump is so obsessed he even has obama‚s name ...,trump is so obsessed he even has obama name co...
4,0,pope francis just called out donald trump duri...,pope francis just called out donald trump duri...


## Remove stopwords

In [15]:
# Create a function to remove stopwords
def remove_stopwords(text):
    processed_docs = []
    stop_words = set(stopwords.words('english'))
    tokens = word_tokenize(text)

    for word in tokens:
        if word not in stop_words:
            processed_docs.append(word)
    return ' '.join(processed_docs)

In [18]:
data_clean['text_no_stopwords'] = data_clean['text_clean'].apply(remove_stopwords)
data_clean.head()

/var/folders/ns/tz5d_bp51j59j7fmbrqg1d1h0000gn/T/ipykernel_20681/4167281436.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_clean['text_no_stopwords'] = data_clean['text_clean'].apply(remove_stopwords)


,label,text,text_clean,text_no_stopwords
0,0,donald trump sends out embarrassing new year‚s...,donald trump sends out embarrassing new year e...,donald trump sends embarrassing new year eve m...
1,0,drunk bragging trump staffer started russian c...,drunk bragging trump staffer started russian c...,drunk bragging trump staffer started russian c...
2,0,sheriff david clarke becomes an internet joke ...,sheriff david clarke becomes an internet joke ...,sheriff david clarke becomes internet joke thr...
3,0,trump is so obsessed he even has obama‚s name ...,trump is so obsessed he even has obama name co...,trump obsessed even obama name coded website i...
4,0,pope francis just called out donald trump duri...,pope francis just called out donald trump duri...,pope francis called donald trump christmas speech


## Lemmatization

In [19]:
def get_wordnet_pos(text):
    tag = nltk.pos_tag([text])[0][1][0]
    tag_dict = {"J": wordnet.ADJ,
                "N": wordnet.NOUN,
                "V": wordnet.VERB,
                "R": wordnet.ADV}
    
    return tag_dict.get(tag, wordnet.NOUN)

In [21]:
wordnet_lemma  = WordNetLemmatizer()

def lemmatize_text(text):
    tokens = word_tokenize(text)
    pos_tags = nltk.pos_tag(tokens)
    lemmatized = [wordnet_lemma.lemmatize(word, get_wordnet_pos(tag)) for word, tag in pos_tags]
    return ' '.join(lemmatized)

In [22]:
data_clean['text_lemmatized'] = data_clean['text_no_stopwords'].apply(lemmatize_text)
data_clean.head()

/var/folders/ns/tz5d_bp51j59j7fmbrqg1d1h0000gn/T/ipykernel_20681/2181263826.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_clean['text_lemmatized'] = data_clean['text_no_stopwords'].apply(lemmatize_text)


,label,text,text_clean,text_no_stopwords,text_lemmatized
0,0,donald trump sends out embarrassing new year‚s...,donald trump sends out embarrassing new year e...,donald trump sends embarrassing new year eve m...,donald trump sends embarrassing new year eve m...
1,0,drunk bragging trump staffer started russian c...,drunk bragging trump staffer started russian c...,drunk bragging trump staffer started russian c...,drunk bragging trump staffer started russian c...
2,0,sheriff david clarke becomes an internet joke ...,sheriff david clarke becomes an internet joke ...,sheriff david clarke becomes internet joke thr...,sheriff david clarke becomes internet joke thr...
3,0,trump is so obsessed he even has obama‚s name ...,trump is so obsessed he even has obama name co...,trump obsessed even obama name coded website i...,trump obsessed even obama name coded website i...
4,0,pope francis just called out donald trump duri...,pope francis just called out donald trump duri...,pope francis called donald trump christmas speech,pope francis called donald trump christmas speech


In [ ]:
data_clean.to_csv("dataset/cleaned_training_data.csv", index=False)

## Split the dataset

In [26]:
# Split into features and labels

X = data_clean['text_lemmatized']
y = data_clean['label']
print(X)
print(y)

0        donald trump sends embarrassing new year eve m...
1        drunk bragging trump staffer started russian c...
2        sheriff david clarke becomes internet joke thr...
3        trump obsessed even obama name coded website i...
4        pope francis called donald trump christmas speech
                               ...                        
34147              tear rain thai gather late king funeral
34148    pyongyang university need non teacher travel b...
34149    philippine president duterte visit japan ahead...
34150                  japan abe may election many want pm
34151    demoralized divided inside catalonia police force
Name: text_lemmatized, Length: 32206, dtype: object
0        0
1        0
2        0
3        0
4        0
        ..
34147    1
34148    1
34149    1
34150    1
34151    1
Name: label, Length: 32206, dtype: int64


In [27]:
# Split data into training and testing set

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X original shape: {X.shape}")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

print(f"y original shape: {y.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X original shape: (32206,)
X_train shape: (25764,)
X_test shape: (6442,)
y original shape: (32206,)
y_train shape: (25764,)
y_test shape: (6442,)


## Bag of Words

In [28]:
vectorizer = CountVectorizer()
X_train_bow = vectorizer.fit_transform(X_train)
X_test_bow = vectorizer.transform(X_test)

print("Vocabulary size:", len(vectorizer.vocabulary_))
print("Feature names:", vectorizer.get_feature_names_out())
print("Matrix shape:", X_train_bow.shape)
print("Matrix type:", type(X_train_bow))
print("Document-Term Matrix:\n", X_train_bow.toarray())

Vocabulary size: 14837
Feature names: ['00' '0045' '0111' ... 'zuma' 'zummar' 'zurich']
Matrix shape: (25764, 14837)
Matrix type: <class 'scipy.sparse._csr.csr_matrix'>
Document-Term Matrix:
 [[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [35]:
# L2 (Euclidean) vector normalization

from sklearn.preprocessing import Normalizer

normalizer = Normalizer(norm="l2")

X_train_bow_normalized = normalizer.fit_transform(X_train_bow)
X_test_bow_normalized = normalizer.transform(X_test_bow)

print(X_train_bow)
print(X_train_bow_normalized)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 221820 stored elements and shape (25764, 14837)>
  Coords	Values
  (0, 7226)	1
  (0, 10505)	1
  (0, 7182)	1
  (0, 1768)	1
  (0, 5880)	1
  (0, 1145)	1
  (1, 2912)	1
  (1, 13467)	1
  (1, 2724)	2
  (1, 9998)	1
  (1, 5697)	1
  (1, 12140)	1
  (2, 7583)	1
  (2, 3284)	1
  (2, 4593)	1
  (2, 7126)	1
  (2, 3780)	1
  (2, 11443)	1
  (3, 13294)	1
  (3, 4251)	1
  (3, 634)	1
  (3, 12112)	1
  (3, 9684)	1
  (3, 14428)	1
  (3, 5920)	1
  :	:
  (25761, 9091)	1
  (25761, 458)	1
  (25761, 3911)	1
  (25761, 14348)	1
  (25761, 7577)	1
  (25761, 12798)	1
  (25761, 10320)	1
  (25761, 12693)	1
  (25761, 9600)	1
  (25762, 14213)	1
  (25762, 2668)	1
  (25762, 11444)	2
  (25762, 10894)	1
  (25762, 13041)	1
  (25762, 7206)	1
  (25762, 7312)	1
  (25762, 11083)	1
  (25762, 4336)	1
  (25762, 4139)	1
  (25763, 9170)	1
  (25763, 2734)	1
  (25763, 2240)	1
  (25763, 13697)	1
  (25763, 3048)	1
  (25763, 5482)	1
<Compressed Sparse Row sparse matrix of dtype 'float64

## TF-IDF

In [29]:
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Print the shape of the dataset
print("X_train_tfidf shape:", X_train_tfidf.shape)
print("X_test_tfidf shape: ", X_test_tfidf.shape)

X_train_tfidf shape: (25764, 14837)
X_test_tfidf shape:  (6442, 14837)


## Baseline classifiers

For this project, the following classifiers will be used:

* Random Forest Classifier
* XGBoost Classifier
* MultinomialNB Classifier

### Random Forest Classifier

In [43]:
# Bag of Words
rf_bow = RandomForestClassifier(criterion='entropy')

rf_bow.fit(X_train_bow_normalized, y_train)

y_pred_rf_bow = rf_bow.predict(X_test_bow_normalized)

print("RandomForest Classifier with CountVectorizer")
print("Accuracy:", accuracy_score(y_test, y_pred_rf_bow))
print(classification_report(y_test, y_pred_rf_bow))
print(confusion_matrix(y_test, y_pred_rf_bow))

# Bag of Words
rf_tfidf = RandomForestClassifier(criterion='entropy')

rf_tfidf.fit(X_train_tfidf, y_train)

y_pred_rf_tfidf = rf_tfidf.predict(X_test_tfidf)

print("RandomForest Classifier with TF-IDF")
print("Accuracy:", accuracy_score(y_test, y_pred_rf_tfidf))
print(classification_report(y_test, y_pred_rf_tfidf))
print(confusion_matrix(y_test, y_pred_rf_tfidf))

RandomForest Classifier with CountVectorizer
Accuracy: 0.9126047811238746
              precision    recall  f1-score   support

           0       0.92      0.91      0.91      3233
           1       0.91      0.92      0.91      3209

    accuracy                           0.91      6442
   macro avg       0.91      0.91      0.91      6442
weighted avg       0.91      0.91      0.91      6442

[[2934  299]
 [ 264 2945]]
RandomForest Classifier with TF-IDF
Accuracy: 0.9098106178205526
              precision    recall  f1-score   support

           0       0.91      0.91      0.91      3233
           1       0.91      0.91      0.91      3209

    accuracy                           0.91      6442
   macro avg       0.91      0.91      0.91      6442
weighted avg       0.91      0.91      0.91      6442

[[2945  288]
 [ 293 2916]]


### MultinomialNB

In [44]:
# Bag of Words

nb_count_bow = MultinomialNB()

nb_count_bow.fit(X_train_bow_normalized, y_train) 

y_pred_nb_count_bow = nb_count_bow.predict(X_test_bow_normalized)

print("MultinomialNB with CountVectorizer")
print("Accuracy:", accuracy_score(y_test, y_pred_nb_count_bow))
print(classification_report(y_test, y_pred_nb_count_bow))
print(confusion_matrix(y_test, y_pred_nb_count_bow))

# TF-IDF 

nb_tfidf = MultinomialNB()

nb_tfidf.fit(X_train_tfidf, y_train)

y_pred_nb_tfidf = nb_tfidf.predict(X_test_tfidf)

print("MultinomialNB with TF-IDF")
print("Accuracy:", accuracy_score(y_test, y_pred_nb_tfidf))
print(classification_report(y_test, y_pred_nb_tfidf))
print(confusion_matrix(y_test, y_pred_nb_tfidf))

MultinomialNB with CountVectorizer
Accuracy: 0.9276622167028873
              precision    recall  f1-score   support

           0       0.92      0.93      0.93      3233
           1       0.93      0.92      0.93      3209

    accuracy                           0.93      6442
   macro avg       0.93      0.93      0.93      6442
weighted avg       0.93      0.93      0.93      6442

[[3014  219]
 [ 247 2962]]
MultinomialNB with TF-IDF
Accuracy: 0.9265755976404844
              precision    recall  f1-score   support

           0       0.92      0.93      0.93      3233
           1       0.93      0.92      0.93      3209

    accuracy                           0.93      6442
   macro avg       0.93      0.93      0.93      6442
weighted avg       0.93      0.93      0.93      6442

[[3019  214]
 [ 259 2950]]


### XGBoost

In [41]:
from xgboost import XGBClassifier

In [46]:
# Bag of Words
xgb_bow = XGBClassifier(eval_metric="logloss")

xgb_bow.fit(X_train_bow_normalized, y_train)

y_pred_xgb_bow = xgb_bow.predict(X_test_bow_normalized)

print("XGBoost with CountVectorizer")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb_bow))
print(classification_report(y_test, y_pred_xgb_bow))
print(confusion_matrix(y_test, y_pred_xgb_bow))

# TF-IDF
xgb_tfidf = XGBClassifier(eval_metric="logloss")

xgb_tfidf.fit(X_train_tfidf, y_train)

y_pred_xgb_tfidf = xgb_tfidf.predict(X_test_tfidf)

print("XGBoost with TF-IDF")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb_tfidf))
print(classification_report(y_test, y_pred_xgb_tfidf))
print(confusion_matrix(y_test, y_pred_xgb_tfidf))

XGBoost with CountVectorizer
Accuracy: 0.888854393045638
              precision    recall  f1-score   support

           0       0.93      0.84      0.88      3233
           1       0.85      0.94      0.89      3209

    accuracy                           0.89      6442
   macro avg       0.89      0.89      0.89      6442
weighted avg       0.89      0.89      0.89      6442

[[2706  527]
 [ 189 3020]]
XGBoost with TF-IDF
Accuracy: 0.8793852840732692
              precision    recall  f1-score   support

           0       0.93      0.82      0.87      3233
           1       0.84      0.94      0.89      3209

    accuracy                           0.88      6442
   macro avg       0.88      0.88      0.88      6442
weighted avg       0.88      0.88      0.88      6442

[[2659  574]
 [ 203 3006]]


In all models, the bag of words vectorizing achieved a higher accuracy, which was more proeminent on XGBoost.

Considering this, the optimization of the classifiers will use CountVectorizer instead of TF-IDF.